<a href="https://colab.research.google.com/github/JORGEDAVI90/PROGCOM-B/blob/main/Piedra_papel_y_tijera.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#PIEDRA PAPEL Y TIJERA

In [2]:
!pip -q install ipywidgets
import random
from collections import Counter
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

ELEMENTOS = [
    ("Piedra", "🪨"),
    ("Papel", "📄"),
    ("Tijera", "✂️"),
    ("Pistola", "🔫"),
    ("Trueno", "⚡"),
    ("Diablo", "😈"),
    ("Dragón", "🐉"),
    ("Agua", "💧"),
    ("Aire", "🌪️"),
    ("Esponja", "🧽"),
    ("Lobo", "🐺"),
    ("Árbol", "🌳"),
    ("Humano", "🧑"),
    ("Serpiente", "🐍"),
    ("Fuego", "🔥"),
]
NOMBRES = [x[0] for x in ELEMENTOS]
EMOJI = {nombre: emoji for nombre, emoji in ELEMENTOS}

AVATARES = [
    ("Ninja", "🥷"),
    ("Astronauta", "🧑‍🚀"),
    ("Mago", "🧙"),
    ("Héroe", "🦸"),
    ("Vampiro", "🧛"),
    ("Sirena", "🧜"),
    ("Hada", "🧚"),
    ("Vaquero", "🤠"),
    ("Alien", "👽"),
    ("Zorro", "🦊"),
    ("Gato", "🐱"),
    ("Panda", "🐼"),
    ("León", "🦁"),
    ("Rana", "🐸"),
    ("Unicornio", "🦄"),
    ("Rey", "👑"),
]
AVATAR_MAQUINA = "🤖"

REGLAS = {}
for i, elemento in enumerate(NOMBRES):
    REGLAS[elemento] = [
        NOMBRES[(i + j) % len(NOMBRES)]
        for j in range(1, 8)
    ]

EXPLICACIONES = {
    ("Agua", "Fuego"): "El agua apaga el fuego.",
    ("Fuego", "Árbol"): "El fuego quema el árbol.",
    ("Piedra", "Tijera"): "La piedra rompe la tijera.",
    ("Papel", "Piedra"): "El papel envuelve la piedra.",
    ("Tijera", "Papel"): "La tijera corta el papel.",
    ("Pistola", "Lobo"): "La pistola derrota al lobo.",
    ("Trueno", "Humano"): "El trueno electrocuta al humano.",
    ("Dragón", "Humano"): "El dragón supera al humano.",
    ("Esponja", "Agua"): "La esponja absorbe el agua.",
    ("Lobo", "Humano"): "El lobo ataca al humano.",
    ("Humano", "Árbol"): "El humano puede talar el árbol.",
    ("Serpiente", "Humano"): "La serpiente puede morder al humano.",
}

objetivo = 5
modo = None
dificultad = "Fácil"
puntos1 = 0
puntos2 = 0
historial = []
historial_usuario = []
eleccion_j1 = None
partida_terminada = False
avatar1 = "👤"
avatar2 = "👤"

CSS = """
<style>
:root {
    --fondo1: #090b18;
    --fondo2: #17152f;
    --morado: #8b5cf6;
    --azul: #38bdf8;
    --rosa: #ec4899;
    --verde: #22c55e;
    --amarillo: #facc15;
}
.game {
    max-width: 1050px;
    margin: 15px auto;
    padding: 30px;
    border-radius: 30px;
    background:
        radial-gradient(circle at 10% 10%, #312e81 0%, transparent 30%),
        radial-gradient(circle at 90% 90%, #701a75 0%, transparent 30%),
        linear-gradient(135deg,var(--fondo1),var(--fondo2));
    color: white;
    font-family: Arial, Helvetica, sans-serif;
    box-shadow: 0 25px 70px rgba(0,0,0,.5);
}
.titulo {
    font-size: 42px;
    font-weight: 900;
    text-align: center;
    margin-bottom: 5px;
    background: linear-gradient(90deg,#38bdf8,#a78bfa,#f472b6);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.subtitulo {
    text-align:center;
    color:#cbd5e1;
    font-size:17px;
    margin-bottom:25px;
}
.panel {
    background:rgba(255,255,255,.07);
    border:1px solid rgba(255,255,255,.12);
    backdrop-filter:blur(15px);
    padding:22px;
    border-radius:22px;
    margin:15px 0;
}
.marcador {
    display:flex;
    gap:20px;
    justify-content:center;
    align-items:center;
}
.jugador {
    flex:1;
    text-align:center;
    background:rgba(255,255,255,.06);
    border-radius:20px;
    padding:18px;
}
.avatar-marcador {
    font-size:52px;
    display:block;
    animation:flotar 1.8s ease-in-out infinite alternate;
}
.avatar-batalla {
    font-size:46px;
    display:block;
    margin-bottom:6px;
}
.avatar-elegido {
    font-size:110px;
    text-align:center;
    animation:flotar 1.4s ease-in-out infinite alternate;
}
.puntos {
    font-size:36px;
    font-weight:900;
}
.vs {
    font-size:30px;
    font-weight:900;
    color:#facc15;
}
.progress {
    height:12px;
    background:#27272a;
    border-radius:999px;
    overflow:hidden;
    margin-top:10px;
}
.progress-bar {
    height:100%;
    background:linear-gradient(90deg,#38bdf8,#8b5cf6,#ec4899);
    border-radius:999px;
    transition:width .5s ease;
}
.elementos {
    display:grid;
    grid-template-columns:repeat(5,1fr);
    gap:12px;
    margin-top:20px;
}
.elemento {
    border:1px solid rgba(255,255,255,.13);
    background:rgba(255,255,255,.07);
    border-radius:18px;
    padding:15px 8px;
    text-align:center;
    cursor:pointer;
    transition:.22s ease;
    user-select:none;
}
.elemento:hover {
    transform:translateY(-6px) scale(1.04);
    background:linear-gradient(135deg,#7c3aed,#2563eb);
    box-shadow:0 10px 25px rgba(124,58,237,.4);
}
.emoji {
    font-size:38px;
    display:block;
    margin-bottom:5px;
}
.nombre {
    font-weight:700;
    font-size:13px;
}
.batalla {
    display:flex;
    justify-content:center;
    align-items:center;
    gap:45px;
    padding:30px 10px;
}
.carta-batalla {
    min-width:180px;
    text-align:center;
    padding:22px;
    border-radius:25px;
    background:rgba(255,255,255,.08);
    animation:entrada .6s ease;
}
.emoji-grande {
    font-size:80px;
    animation:flotar 1.6s ease-in-out infinite alternate;
}
.choque {
    font-size:50px;
    animation:choque .45s infinite alternate;
}
.resultado {
    font-size:27px;
    font-weight:900;
    text-align:center;
    margin:15px;
}
.historial {
    max-height:250px;
    overflow-y:auto;
}
.ronda {
    padding:10px;
    border-bottom:1px solid rgba(255,255,255,.1);
}
.regla {
    padding:12px;
    margin:8px 0;
    border-radius:12px;
    background:rgba(255,255,255,.05);
}
.cuenta {
    text-align:center;
    font-size:75px;
    font-weight:900;
    animation:zoom .8s ease infinite alternate;
}
.final {
    text-align:center;
    padding:35px;
}
.trofeo {
    font-size:100px;
    animation:trofeo 1s ease-in-out infinite alternate;
}
@keyframes entrada {
    from {opacity:0; transform:scale(.5);}
    to {opacity:1; transform:scale(1);}
}
@keyframes flotar {
    from {transform:translateY(-5px) rotate(-4deg);}
    to {transform:translateY(7px) rotate(4deg);}
}
@keyframes choque {
    from {transform:scale(.8) rotate(-8deg);}
    to {transform:scale(1.25) rotate(8deg);}
}
@keyframes zoom {
    from {transform:scale(.75); opacity:.6;}
    to {transform:scale(1.2); opacity:1;}
}
@keyframes trofeo {
    from {transform:rotate(-7deg) scale(.95);}
    to {transform:rotate(7deg) scale(1.08);}
}
.confetti {
    position:fixed;
    top:-20px;
    font-size:25px;
    z-index:9999;
    pointer-events:none;
    animation:caer 3s linear forwards;
}
@keyframes caer {
    to {
        transform:translateY(110vh) rotate(900deg);
        opacity:0;
    }
}
@media(max-width:700px){
    .elementos {
        grid-template-columns:repeat(3,1fr);
    }
    .titulo {
        font-size:30px;
    }
    .batalla {
        gap:12px;
    }
    .emoji-grande {
        font-size:55px;
    }
}
</style>
"""
display(HTML(CSS))

pantalla = widgets.Output()
display(pantalla)


def boton(texto, color="#7c3aed", ancho="220px"):
    """Crea botones estilizados de ipywidgets."""
    b = widgets.Button(
        description=texto,
        layout=widgets.Layout(
            width=ancho,
            height="48px",
            margin="6px"
        )
    )
    b.style.button_color = color
    b.style.font_weight = "bold"
    return b


def html(contenido):
    display(HTML(contenido))


def nombre_jugador1():
    return "Jugador 1"


def nombre_jugador2():
    if modo == "Máquina":
        return "Máquina"
    return "Jugador 2"


def barra(valor):
    porcentaje = min(100, (valor / objetivo) * 100)
    return f"""
    <div class="progress">
        <div class="progress-bar" style="width:{porcentaje}%"></div>
    </div>
    """


def mostrar_marcador():
    html(f"""
    <div class="panel marcador">
        <div class="jugador">
            <span class="avatar-marcador">{avatar1}</span>
            <b>{nombre_jugador1()}</b>
            <div class="puntos">{puntos1}</div>
            {barra(puntos1)}
        </div>
        <div class="vs">VS</div>
        <div class="jugador">
            <span class="avatar-marcador">{avatar2}</span>
            <b>{nombre_jugador2()}</b>
            <div class="puntos">{puntos2}</div>
            {barra(puntos2)}
        </div>
    </div>
    """)


def resultado_ronda(a, b):
    """
    Retorna:
    1 = gana jugador 1
    2 = gana jugador 2
    0 = empate
    """
    if a == b:
        return 0
    if b in REGLAS[a]:
        return 1
    return 2


def descripcion_regla(ganador, perdedor):
    if (ganador, perdedor) in EXPLICACIONES:
        return EXPLICACIONES[(ganador, perdedor)]
    return (
        f"{EMOJI[ganador]} {ganador} supera a "
        f"{EMOJI[perdedor]} {perdedor}."
    )


def elegir_maquina():
    """Selecciona la jugada de la máquina según dificultad."""
    if dificultad == "Fácil":
        return random.choice(NOMBRES)

    if dificultad == "Media":
        if historial_usuario and random.random() < 0.55:
            ultima = historial_usuario[-1]
            counters = [
                x for x in NOMBRES
                if ultima in REGLAS[x]
            ]
            return random.choice(counters)
        return random.choice(NOMBRES)

    if not historial_usuario:
        return random.choice(NOMBRES)

    frecuencia = Counter(historial_usuario)
    mas_usado = frecuencia.most_common(1)[0][0]
    prediccion = mas_usado

    if len(historial_usuario) >= 3:
        ultima = historial_usuario[-1]
        siguientes = []
        for i in range(len(historial_usuario) - 1):
            if historial_usuario[i] == ultima:
                siguientes.append(historial_usuario[i + 1])
        if siguientes:
            prediccion = Counter(siguientes).most_common(1)[0][0]

    counters = [
        x for x in NOMBRES
        if prediccion in REGLAS[x]
    ]

    if random.random() < 0.75:
        return random.choice(counters)
    return random.choice(NOMBRES)


def lanzar_confeti():
    display(HTML("""
    <script>
    (function(){
        const emojis = ["🎉","✨","⭐","🎊","💥","🥳","🏆"];
        for(let i=0;i<55;i++){
            let c = document.createElement("div");
            c.className = "confetti";
            c.innerHTML = emojis[
                Math.floor(Math.random()*emojis.length)
            ];
            c.style.left = Math.random()*100 + "vw";
            c.style.animationDuration =
                (2 + Math.random()*2) + "s";
            document.body.appendChild(c);
            setTimeout(()=>c.remove(),4000);
        }
    })();
    </script>
    """))


def menu_principal():
    global puntos1, puntos2, historial
    global historial_usuario, partida_terminada
    global avatar1, avatar2
    puntos1 = 0
    puntos2 = 0
    historial = []
    historial_usuario = []
    partida_terminada = False
    avatar1 = "👤"
    avatar2 = "👤"
    with pantalla:
        clear_output()
        html("""
        <div class="game">
            <div class="titulo">
                ⚔️ BATALLA DE LOS 15 ELEMENTOS ⚔️
            </div>
            <div class="subtitulo">
                Piedra, Papel o Tijera llevado a otro nivel
            </div>
            <div class="panel" style="text-align:center">
                <h2>🎯 ¿Cuántos puntos se necesitan para ganar?</h2>
                <p>
                Configura la partida y después selecciona
                el modo de juego.
                <br><br>
                <b>15 elementos • avatares • IA • batallas épicas</b>
                </p>
            </div>
        </div>
        """)
        selector = widgets.IntSlider(
            value=5,
            min=1,
            max=20,
            step=1,
            description="Puntos:",
            continuous_update=True
        )
        b_maquina = boton(
            "🤖 Jugador vs Máquina",
            "#2563eb",
            "250px"
        )
        b_dos = boton(
            "👥 Jugador vs Jugador",
            "#9333ea",
            "250px"
        )
        b_reglas = boton(
            "📜 Ver reglas",
            "#475569",
            "180px"
        )
        display(
            widgets.VBox([
                selector,
                widgets.HBox(
                    [b_maquina, b_dos],
                    layout=widgets.Layout(
                        justify_content="center",
                        flex_flow="row wrap"
                    )
                ),
                widgets.HBox(
                    [b_reglas],
                    layout=widgets.Layout(
                        justify_content="center"
                    )
                )
            ])
        )

        def maquina_click(_):
            global objetivo, modo
            objetivo = selector.value
            modo = "Máquina"
            pantalla_dificultad()

        def dos_click(_):
            global objetivo, modo
            objetivo = selector.value
            modo = "Dos jugadores"
            pantalla_avatar(
                1,
                lambda: pantalla_avatar(2, comenzar_partida)
            )

        b_maquina.on_click(maquina_click)
        b_dos.on_click(dos_click)
        b_reglas.on_click(lambda _: mostrar_reglas())


def pantalla_dificultad():
    with pantalla:
        clear_output()
        html("""
        <div class="game">
            <div class="titulo">🤖 NIVEL DE LA MÁQUINA</div>
            <div class="panel" style="text-align:center">
                <h2>Selecciona una dificultad</h2>
                <p>
                Fácil juega al azar.<br>
                Media intenta anticipar tu última jugada.<br>
                Difícil analiza tus hábitos y patrones.
                </p>
            </div>
        </div>
        """)
        facil = boton("😊 Fácil", "#16a34a")
        media = boton("🧠 Media", "#d97706")
        dificil = boton("🔥 Difícil", "#dc2626")
        volver = boton("⬅️ Volver", "#475569")
        display(
            widgets.HBox(
                [facil, media, dificil],
                layout=widgets.Layout(
                    justify_content="center",
                    flex_flow="row wrap"
                )
            )
        )
        display(
            widgets.HBox(
                [volver],
                layout=widgets.Layout(
                    justify_content="center"
                )
            )
        )

        def seleccionar(nivel):
            global dificultad, avatar2
            dificultad = nivel
            avatar2 = AVATAR_MAQUINA
            pantalla_avatar(1, comenzar_partida)

        facil.on_click(lambda _: seleccionar("Fácil"))
        media.on_click(lambda _: seleccionar("Media"))
        dificil.on_click(lambda _: seleccionar("Difícil"))
        volver.on_click(lambda _: menu_principal())


def pantalla_avatar(jugador, siguiente):
    """Pantalla para que cada jugador elija su avatar."""
    with pantalla:
        clear_output()
        if modo == "Máquina":
            titulo = "🎭 ELIGE TU AVATAR"
        else:
            titulo = f"🎭 JUGADOR {jugador}: ELIGE TU AVATAR"
        html(f"""
        <div class="game">
            <div class="titulo">{titulo}</div>
            <div class="panel" style="text-align:center">
                <h2>Escoge el personaje que te representará</h2>
                <p>Aparecerá en el marcador, en las batallas y en la pantalla final.</p>
            </div>
        </div>
        """)

        botones = []
        for nombre, emoji in AVATARES:
            if jugador == 2 and emoji == avatar1:
                continue
            b = widgets.Button(
                description=f"{emoji} {nombre}",
                layout=widgets.Layout(
                    width="175px",
                    height="60px",
                    margin="5px"
                )
            )
            b.style.font_weight = "bold"

            def elegir(_, e=emoji):
                global avatar1, avatar2
                if jugador == 1:
                    avatar1 = e
                else:
                    avatar2 = e
                siguiente()

            b.on_click(elegir)
            botones.append(b)

        display(
            widgets.Box(
                botones,
                layout=widgets.Layout(
                    display="flex",
                    flex_flow="row wrap",
                    justify_content="center"
                )
            )
        )

        volver = boton("⬅️ Volver al menú", "#475569", "200px")
        volver.on_click(lambda _: menu_principal())
        display(
            widgets.HBox(
                [volver],
                layout=widgets.Layout(
                    justify_content="center"
                )
            )
        )


def botones_elementos(callback):
    botones = []
    for nombre, emoji in ELEMENTOS:
        b = widgets.Button(
            description=f"{emoji} {nombre}",
            layout=widgets.Layout(
                width="175px",
                height="60px",
                margin="5px"
            )
        )
        b.style.font_weight = "bold"
        b.on_click(
            lambda _, n=nombre: callback(n)
        )
        botones.append(b)
    return widgets.Box(
        botones,
        layout=widgets.Layout(
            display="flex",
            flex_flow="row wrap",
            justify_content="center"
        )
    )


def comenzar_partida():
    global puntos1, puntos2, historial
    global historial_usuario, eleccion_j1
    global partida_terminada
    puntos1 = 0
    puntos2 = 0
    historial = []
    historial_usuario = []
    eleccion_j1 = None
    partida_terminada = False
    nueva_ronda()


def nueva_ronda():
    if partida_terminada:
        return
    with pantalla:
        clear_output()
        html('<div class="game">')
        html('<div class="titulo">⚔️ ELIGE TU ELEMENTO</div>')
        mostrar_marcador()
        if modo == "Máquina":
            html(f"""
            <div class="panel" style="text-align:center">
                <h3>🤖 Dificultad: {dificultad}</h3>
                <p>
                    {avatar1} Jugador 1: selecciona tu próxima jugada.
                </p>
            </div>
            """)
            display(
                botones_elementos(jugada_contra_maquina)
            )
        else:
            html(f"""
            <div class="panel" style="text-align:center">
                <h3>{avatar1} Turno del Jugador 1</h3>
                <p>
                    Elige en secreto tu elemento.
                </p>
            </div>
            """)
            display(
                botones_elementos(seleccion_jugador1)
            )
        mostrar_historial()
        html('</div>')


def jugada_contra_maquina(eleccion):
    if partida_terminada:
        return
    historial_usuario.append(eleccion)
    maquina = elegir_maquina()
    revelar_batalla(
        eleccion,
        maquina,
        "Jugador 1",
        "Máquina"
    )


def seleccion_jugador1(eleccion):
    global eleccion_j1
    eleccion_j1 = eleccion
    with pantalla:
        clear_output()
        html(f"""
        <div class="game">
            <div class="titulo">
                🔒 ELECCIÓN GUARDADA
            </div>
            <div class="panel" style="text-align:center">
                <div style="font-size:70px">🙈</div>
                <h2>
                    {avatar1} Jugador 1 ya eligió
                </h2>
                <p>
                    Su elección está oculta.
                    <br><br>
                    Pasa el dispositivo al Jugador 2.
                </p>
            </div>
        </div>
        """)
        continuar = boton(
            f"{avatar2} Soy el Jugador 2",
            "#2563eb",
            "240px"
        )
        display(
            widgets.HBox(
                [continuar],
                layout=widgets.Layout(
                    justify_content="center"
                )
            )
        )
        continuar.on_click(
            lambda _: turno_jugador2()
        )


def turno_jugador2():
    with pantalla:
        clear_output()
        html('<div class="game">')
        html(
            f'<div class="titulo">{avatar2} TURNO DEL JUGADOR 2</div>'
        )
        mostrar_marcador()
        html("""
        <div class="panel" style="text-align:center">
            <h3>Selecciona tu elemento</h3>
            <p>
                No podrás ver qué escogió el Jugador 1
                hasta que ambos hayan jugado.
            </p>
        </div>
        """)
        display(
            botones_elementos(seleccion_jugador2)
        )
        html('</div>')


def seleccion_jugador2(eleccion):
    revelar_batalla(
        eleccion_j1,
        eleccion,
        "Jugador 1",
        "Jugador 2"
    )


def revelar_batalla(a, b, nombre1, nombre2):
    with pantalla:
        clear_output()
        html(f"""
        <div class="game">
            <div class="titulo">
                ⚔️ PREPÁRATE...
            </div>
            <div class="panel cuenta">
                {avatar1} 3... 2... 1... {avatar2}
                <br>
                💥
            </div>
        </div>
        """)
    display(HTML("""
    <script>
        console.log("3... 2... 1... BATALLA!");
    </script>
    """))
    resolver_batalla(a, b, nombre1, nombre2)


def resolver_batalla(a, b, nombre1, nombre2):
    global puntos1, puntos2, partida_terminada
    resultado = resultado_ronda(a, b)
    if resultado == 1:
        puntos1 += 1
        mensaje = f"🏆 ¡{avatar1} {nombre1} gana la ronda!"
        detalle = descripcion_regla(a, b)
        efecto = "🎉✨🥳"
    elif resultado == 2:
        puntos2 += 1
        mensaje = f"🏆 ¡{avatar2} {nombre2} gana la ronda!"
        detalle = descripcion_regla(b, a)
        if modo == "Máquina":
            efecto = "😭💔🤖"
        else:
            efecto = "🎉✨🥳"
    else:
        mensaje = "🤝 ¡EMPATE!"
        detalle = "Ambos eligieron el mismo elemento."
        efecto = "😐⚔️😐"
    historial.append(
        {
            "a": a,
            "b": b,
            "resultado": mensaje
        }
    )
    with pantalla:
        clear_output()
        html('<div class="game">')
        html('<div class="titulo">💥 ¡BATALLA!</div>')
        mostrar_marcador()
        html(f"""
        <div class="batalla">
            <div class="carta-batalla">
                <span class="avatar-batalla">{avatar1}</span>
                <div class="emoji-grande">
                    {EMOJI[a]}
                </div>
                <h3>{nombre1}</h3>
                <b>{a}</b>
            </div>
            <div class="choque">
                ⚡VS⚡
            </div>
            <div class="carta-batalla">
                <span class="avatar-batalla">{avatar2}</span>
                <div class="emoji-grande">
                    {EMOJI[b]}
                </div>
                <h3>{nombre2}</h3>
                <b>{b}</b>
            </div>
        </div>
        <div class="resultado">
            {mensaje}
        </div>
        <div style="
            text-align:center;
            font-size:35px;
            margin:10px;
        ">
            {efecto}
        </div>
        <div class="panel" style="text-align:center">
            {detalle}
        </div>
        """)
        if puntos1 >= objetivo or puntos2 >= objetivo:
            partida_terminada = True
            siguiente = boton(
                "🏆 Ver resultado final",
                "#eab308",
                "250px"
            )
            display(
                widgets.HBox(
                    [siguiente],
                    layout=widgets.Layout(
                        justify_content="center"
                    )
                )
            )
            siguiente.on_click(
                lambda _: pantalla_final()
            )
            lanzar_confeti()
        else:
            siguiente = boton(
                "⚔️ Siguiente ronda",
                "#7c3aed",
                "230px"
            )
            reglas = boton(
                "📜 Ver reglas",
                "#475569",
                "180px"
            )
            display(
                widgets.HBox(
                    [siguiente, reglas],
                    layout=widgets.Layout(
                        justify_content="center",
                        flex_flow="row wrap"
                    )
                )
            )
            siguiente.on_click(
                lambda _: nueva_ronda()
            )
            reglas.on_click(
                lambda _: mostrar_reglas(
                    volver_a="partida"
                )
            )
        mostrar_historial()
        html('</div>')


def mostrar_historial():
    if not historial:
        return
    contenido = ""
    ultimas = historial[-8:]
    for numero, ronda in enumerate(
        ultimas,
        start=max(1, len(historial)-len(ultimas)+1)
    ):
        contenido += f"""
        <div class="ronda">
            <b>Ronda {numero}</b> ·
            {EMOJI[ronda['a']]} {ronda['a']}
            ⚔️
            {EMOJI[ronda['b']]} {ronda['b']}
            <br>
            <small>{ronda['resultado']}</small>
        </div>
        """
    html(f"""
    <div class="panel">
        <h3>📜 Historial reciente</h3>
        <div class="historial">
            {contenido}
        </div>
    </div>
    """)


def mostrar_reglas(volver_a="menu"):
    with pantalla:
        clear_output()
        html("""
        <div class="game">
        <div class="titulo">
            📜 REGLAS DE LOS 15 ELEMENTOS
        </div>
        <div class="panel">
        <p style="text-align:center">
            Cada elemento vence exactamente a
            <b>7 elementos</b> y pierde contra otros
            <b>7</b>.
        </p>
        """)
        for elemento in NOMBRES:
            victimas = " · ".join(
                f"{EMOJI[x]} {x}"
                for x in REGLAS[elemento]
            )
            html(f"""
            <div class="regla">
                <b>
                    {EMOJI[elemento]} {elemento}
                </b>
                vence a:
                <br><br>
                {victimas}
            </div>
            """)
        html("""
        </div>
        </div>
        """)
        volver = boton(
            "⬅️ Volver",
            "#475569",
            "180px"
        )
        display(
            widgets.HBox(
                [volver],
                layout=widgets.Layout(
                    justify_content="center"
                )
            )
        )
        if volver_a == "partida":
            volver.on_click(
                lambda _: nueva_ronda()
            )
        else:
            volver.on_click(
                lambda _: menu_principal()
            )


def pantalla_final():
    if puntos1 > puntos2:
        ganador = "Jugador 1"
        avatar_ganador = avatar1
    elif modo == "Máquina":
        ganador = "Máquina"
        avatar_ganador = avatar2
    else:
        ganador = "Jugador 2"
        avatar_ganador = avatar2

    if ganador == "Jugador 1":
        emoji_final = "🏆🥳🎉"
        titulo_final = "¡VICTORIA!"
    elif ganador == "Máquina":
        emoji_final = "🤖🏆💀"
        titulo_final = "¡LA MÁQUINA GANÓ!"
    else:
        emoji_final = "🏆🎉🥳"
        titulo_final = "¡VICTORIA!"
    with pantalla:
        clear_output()
        html(f"""
        <div class="game">
            <div class="final">
                <div class="trofeo">
                    🏆
                </div>
                <div class="avatar-elegido">
                    {avatar_ganador}
                </div>
                <div class="titulo">
                    {titulo_final}
                </div>
                <h1>
                    {emoji_final}
                </h1>
                <h2>
                    {avatar_ganador} {ganador} gana la partida
                </h2>
                <div class="panel">
                    <h2>
                        MARCADOR FINAL
                    </h2>
                    <div style="
                        font-size:40px;
                        font-weight:900;
                    ">
                        {avatar1} {puntos1}
                        &nbsp; ⚔️ &nbsp;
                        {puntos2} {avatar2}
                    </div>
                    <p>
                        Objetivo: {objetivo} puntos
                    </p>
                </div>
            </div>
        </div>
        """)
        otra = boton(
            "🔄 Jugar de nuevo",
            "#16a34a",
            "220px"
        )
        menu = boton(
            "🏠 Volver al menú",
            "#2563eb",
            "220px"
        )
        reglas = boton(
            "📜 Ver reglas",
            "#475569",
            "180px"
        )
        display(
            widgets.HBox(
                [otra, menu, reglas],
                layout=widgets.Layout(
                    justify_content="center",
                    flex_flow="row wrap"
                )
            )
        )
        otra.on_click(
            lambda _: comenzar_partida()
        )
        menu.on_click(
            lambda _: menu_principal()
        )
        reglas.on_click(
            lambda _: mostrar_reglas()
        )
        lanzar_confeti()


menu_principal()

Output()